In [22]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
netflix = pd.read_csv("Dataset.csv")
netflix.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## ****Data Exploration****
    1. Here we are going to explore dataset like getting information of all columns in dataset checking if they are required or redudunt for project.
    2. Also check if any duplicate recoreds or missing reords are present. if present then removing them

In [3]:
netflix.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


## ****Feature Engineering****
- Select the required columns from whole dataset like show_id, type, title, director, listed_in, release_year.
- what column describes:
    1. **show_id**: Unique id of each show.
    2. **type**: Type of show 'movie' or 'tv show'.
    3. **director**: name of director of related show
    4. **listed_in**: genre of show
    5. **release_year**: year in which show released

- Data Cleaning and feature construction
    1. converted **type** column text in lower case.
    2. converted **director** column text in lower case as well as removed space between name.
    3. Removed repeated words in **listed_in** column like 'TV', 'shows', 'movies' to capture unique words for genre
    4. Merged **type**, **director**, **listed_in** columns in single column and named it as tags.

- The **tags** column significantly enhances similarity-based recommendations by enabling precise semantic matching between user interests and item attributes.

In [4]:
# show_id 
# type 
# title  
# director 
# listed_in
# release_year

In [5]:
df = netflix[["show_id", "type", "title", "director", "listed_in", "release_year"]]
df

,show_id,type,title,director,listed_in,release_year
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Documentaries,2020
1,s3,TV Show,Ganglands,Julien Leclercq,"Crime TV Shows, International TV Shows, TV Act...",2021
2,s6,TV Show,Midnight Mass,Mike Flanagan,"TV Dramas, TV Horror, TV Mysteries",2021
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,"Children & Family Movies, Comedies",2021
4,s8,Movie,Sankofa,Haile Gerima,"Dramas, Independent Movies, International Movies",1993
...,...,...,...,...,...,...
8785,s8797,TV Show,Yunus Emre,Not Given,"International TV Shows, TV Dramas",2016
8786,s8798,TV Show,Zak Storm,Not Given,Kids' TV,2016
8787,s8801,TV Show,Zindagi Gulzar Hai,Not Given,"International TV Shows, Romantic TV Shows, TV ...",2012
8788,s8784,TV Show,Yoko,Not Given,Kids' TV,2016


In [6]:
def create_tags(dfa: pd.DataFrame) -> pd.DataFrame:
    """This function return a dataset after cleaning it and adding new coumn named tags
    tags is concatination of type, director, listed_in, rating columns."""

    df = dfa[["show_id", "type", "title", "director", "listed_in", "release_year", "rating"]]
    
    df["type"] = df["type"].str.strip().str.replace(" ","").str.lower()
    df["director"] = df["director"].str.strip().str.replace("Not Given", "Unknown").str.replace(" ", "").str.lower()
    df["listed_in"] = df["listed_in"].str.strip().str.lower().str.replace("movies", "").str.replace("tv", "").str.replace("shows", "").str.replace("  ", " ").str.replace(" , ", ", ")
    df["rating"] = df["rating"].str.lower().str.strip().str.replace("-", "_")

    df["tags"] = df["listed_in"] +" "+ df["type"] +" "+ df["director"] +" "+ df["rating"]
    df = df.drop(columns = ["listed_in", "type", "director", "rating"])
    return df


df = create_tags(netflix)

In [16]:
ps = PorterStemmer()
def stem(text):
    words = []

    for i in text.split():
        words.append(ps.stem(i))

    return " ".join(words)

In [17]:
df["tags"] = df['tags'].apply(stem)
df

,show_id,title,release_year,tags
0,s1,Dick Johnson Is Dead,2020,documentari movi kirstenjohnson pg_13
1,s3,Ganglands,2021,"crime, international, action & adventur tvshow..."
2,s6,Midnight Mass,2021,"dramas, horror, mysteri tvshow mikeflanagan tv_ma"
3,s14,Confessions of an Invisible Girl,2021,"children & family, comedi movi brunogarotti tv_pg"
4,s8,Sankofa,1993,"dramas, independent, intern movi hailegerima t..."
...,...,...,...,...
8785,s8797,Yunus Emre,2016,"international, drama tvshow unknown tv_pg"
8786,s8798,Zak Storm,2016,kids' tvshow unknown tv_y7
8787,s8801,Zindagi Gulzar Hai,2012,"international, romantic, drama tvshow unknown ..."
8788,s8784,Yoko,2016,kids' tvshow unknown tv_i


In [23]:
cv = CountVectorizer(max_features=5000, stop_words="english")
vectors = cv.fit_transform(df["tags"]).toarray()

In [25]:
similarity = cosine_similarity(vectors)

In [50]:
sl = sorted(list(enumerate(similarity[1])), reverse=True, key= lambda x: x[1])[1:6]

for i in sl:
    print(df.iloc[i[0]].title)

Bangkok Breaking
Fatal Destiny
Nowhere Man
Undercover
Lupin


In [53]:
def recommend(movie):
    movie_index = df[df["title"] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(enumerate(distances), reverse=True, key=lambda x: x[1])[1:6]
    for i in movies_list:
        print(df.iloc[i[0]].title)

In [ ]:
def recommend_year(movie):
    movie_index = df[df["title"] == movie].index[0]
    fetch_year = df.loc[movie_index, "release_year"]
    

64                                        Safe House
278                         Kyaa Super Kool Hain Hum
405                                          Quartet
430         American Masters: Inventing David Geffen
434    The Great British Baking Show: The Beginnings
Name: title, dtype: str